In [1]:

# ============================================================
# FE5108 Final Project - Part 1
# Question 1D: Sensitivity to Factor Portfolio Construction
# ============================================================

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Locate the repository root
current_dir = Path.cwd().resolve()
ROOT = next(
    (p for p in [current_dir, *current_dir.parents]
     if (p / "src" / "config.py").exists()),
    None
)

if ROOT is None:
    raise FileNotFoundError("Project root not found.")

sys.path.insert(0, str(ROOT))

from src import config, utils

config.apply_style()

# Load the previously validated monthly return series
msci_returns = pd.read_csv(
    config.TAB_DIR / "tab_1C_monthly_returns.csv",
    index_col="month"
)

factor_spreads = pd.read_csv(
    config.TAB_DIR / "tab_1A_factor_spreads.csv",
    index_col="month"
)

# Convert indexes to monthly periods
msci_returns.index = pd.PeriodIndex(
    msci_returns.index, freq="M"
)

factor_spreads.index = pd.PeriodIndex(
    factor_spreads.index, freq="M"
)

print("MSCI returns:", msci_returns.shape)
print("Factor spreads:", factor_spreads.shape)

display(msci_returns.head(3))
display(factor_spreads.head(3))


MSCI returns: (343, 6)
Factor spreads: (343, 2)


,Market,Value,MinVol,EqualWeight,Momentum,MultiFactor
month,,,,,,
1995-01,0.0278,0.0297,0.0225,0.0241,0.0127,0.0222
1995-02,0.0397,0.0415,0.0363,0.0454,0.0415,0.0412
1995-03,0.0270,0.0264,0.0297,0.0277,0.0459,0.0324


,Momentum,LowMinusHighBeta
month,,
1995-01,-0.0346,0.0122
1995-02,0.0114,-0.0272
1995-03,0.0360,0.0063


In [2]:

# Step 2: Construct MSCI factor return spreads over the market

msci_spreads = pd.DataFrame(
    index=msci_returns.index
)

msci_spreads["Value"] = (
    msci_returns["Value"] - msci_returns["Market"]
)

msci_spreads["MinVol"] = (
    msci_returns["MinVol"] - msci_returns["Market"]
)

msci_spreads["EqualWeight"] = (
    msci_returns["EqualWeight"] - msci_returns["Market"]
)

msci_spreads["Momentum"] = (
    msci_returns["Momentum"] - msci_returns["Market"]
)

msci_spreads = utils.select_period(
    msci_spreads, "1D"
)

display(msci_spreads.head())

print("Sample:", msci_spreads.index.min(),
      "to", msci_spreads.index.max())


,Value,MinVol,EqualWeight,Momentum
month,,,,
1995-01,0.0019,-0.0053,-0.0037,-0.0151
1995-02,0.0018,-0.0034,0.0057,0.0018
1995-03,-0.0006,0.0027,0.0007,0.0189
1995-04,0.0029,-0.0045,-0.0087,0.0115
1995-05,0.0016,-0.0054,-0.0018,-0.0020


Sample: 1995-01 to 2023-07


In [3]:

# Step 3: Load the monthly Fama-French HML and SMB spreads

raw_ff = pd.read_excel(
    config.FILE_FF_MSCI,
    sheet_name=config.SHEET_FF3,
    header=None
)

ff = raw_ff.iloc[4:].copy()
ff.columns = ["Date", "Mkt-RF", "SMB", "HML", "RF"]

# Keep only monthly observations
date_text = ff["Date"].astype(str).str.strip()
mask = date_text.str.fullmatch(r"\d{6}(\.0)?")

ff = ff.loc[mask].copy()

ff["Date"] = (
    ff["Date"].astype(float).astype(int).astype(str)
)

ff = ff.set_index("Date")
ff = ff.apply(pd.to_numeric, errors="coerce")

# Convert percentages to decimals and standardize dates
ff = utils.pct_to_decimal(ff)
ff = utils.to_monthly_index(ff)
ff = utils.select_period(ff, "1D")

# Combine all four academic-style factor spreads
academic_spreads = pd.DataFrame({
    "HML": ff["HML"],
    "SMB": ff["SMB"],
    "Momentum": factor_spreads["Momentum"],
    "LowBeta": factor_spreads["LowMinusHighBeta"]
})

display(academic_spreads.head())


,HML,SMB,Momentum,LowBeta
month,,,,
1995-01,0.0257,-0.0350,-0.0346,0.0122
1995-02,0.0108,-0.0066,0.0114,-0.0272
1995-03,-0.0215,-0.0016,0.0360,0.0063
1995-04,0.0171,-0.0049,0.0194,0.0309
1995-05,0.0229,-0.0255,-0.0011,-0.0059


In [4]:

# Step 4: Calculate correlations between factor constructions

# Verify consistent dates and complete observations
assert msci_spreads.index.equals(academic_spreads.index)
assert len(msci_spreads) == 343
assert msci_spreads.notna().all().all()
assert academic_spreads.notna().all().all()

# Calculate the 4-by-4 Pearson correlation matrix
correlation_matrix = pd.DataFrame(
    index=msci_spreads.columns,
    columns=academic_spreads.columns,
    dtype=float
)

for msci_factor in msci_spreads.columns:
    for academic_factor in academic_spreads.columns:
        correlation_matrix.loc[
            msci_factor, academic_factor
        ] = msci_spreads[msci_factor].corr(
            academic_spreads[academic_factor]
        )

print("===== QUESTION 1D: CORRELATION MATRIX =====")
display(correlation_matrix.round(3))

# Export the correlation matrix
saved_path = utils.save_table(
    correlation_matrix,
    "tab_1D_correlations"
)

print("Saved:", saved_path)


===== QUESTION 1D: CORRELATION MATRIX =====


,HML,SMB,Momentum,LowBeta
Value,0.728,-0.177,-0.244,0.415
MinVol,0.232,-0.307,0.303,0.815
EqualWeight,0.431,0.322,-0.449,-0.199
Momentum,-0.222,0.079,0.642,0.140


Saved: /Users/zhangjiaxin/Desktop/fe5018-final/output/tables/tab_1D_correlations.csv
